# Stem separation + note detection (fast path)

Runs Demucs and note detection on Colab's free GPU (~seconds to low-minutes
per song, vs 15-40 min on a CPU-only laptop). Output is a zip you extract
directly into your local `backend/data/` folder -- no other setup needed
on the local app side, it picks up new songs automatically.

**Before running:** Runtime menu -> Change runtime type -> GPU (T4 is fine).

**Limitation carried over from the local app:** note detection uses a
monophonic pitch tracker (librosa.pyin). It's reliable on isolated bass
or vocal stems, but the "other" stem is frequently polyphonic (piano +
guitar + synth layered) and note detection on it is best-effort -- expect
it to be wrong on chords. Drums never gets note detection attempted.

In [ ]:
!pip install -q demucs librosa soundfile

## Upload a song

Any common audio format works (mp3, wav, flac...). Run this cell, then use
the file picker that appears.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print(f"Uploaded: {input_filename}")

## Run Demucs separation

In [ ]:
import subprocess
import sys
from pathlib import Path

MODEL_NAME = "htdemucs"
input_path = Path(input_filename)
song_id = input_path.stem.replace(" ", "_")

demucs_out = Path("_demucs_raw")

result = subprocess.run(
    [sys.executable, "-m", "demucs", "-n", MODEL_NAME, "-o", str(demucs_out), str(input_path)],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Demucs failed -- see stderr above")

stems_source_dir = demucs_out / MODEL_NAME / input_path.stem
print(f"Stems written to: {stems_source_dir}")
print(list(stems_source_dir.glob('*.wav')))

## Note detection

This cell is intentionally near-identical to `backend/note_extraction.py`
in the local project -- duplicated rather than imported, since Colab can't
easily reach into your local backend folder. If you tune the extraction
parameters (thresholds, frequency ranges) in one place, update the other
to match, or the local-import path and the Colab path will silently
produce different results for the same song.

In [ ]:
import numpy as np
import librosa
import json

FREQ_RANGES = {
    "bass": ("C1", "G4"),
    "vocals": ("C2", "C6"),
    "other": ("C2", "C6"),
}
MIN_SEGMENT_DURATION = 0.08
NOTE_CHANGE_THRESHOLD_SEMITONES = 0.5

def extract_note_timeline(audio_path, instrument):
    fmin_name, fmax_name = FREQ_RANGES[instrument]
    fmin = librosa.note_to_hz(fmin_name)
    fmax = librosa.note_to_hz(fmax_name)

    y, sr = librosa.load(audio_path, sr=None, mono=True)
    f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=sr)
    times = librosa.times_like(f0, sr=sr)

    raw_points = []
    for t, f, voiced in zip(times, f0, voiced_flag):
        if voiced and f is not None and not np.isnan(f):
            raw_points.append((float(t), librosa.hz_to_midi(float(f))))

    if not raw_points:
        return []

    segments = []
    seg_start_time = raw_points[0][0]
    seg_midi_values = [raw_points[0][1]]
    prev_time = raw_points[0][0]
    frame_hop = times[1] - times[0] if len(times) > 1 else 0.01
    gap_threshold = frame_hop * 3

    for t, midi in raw_points[1:]:
        running_avg = sum(seg_midi_values) / len(seg_midi_values)
        pitch_close = abs(midi - running_avg) <= NOTE_CHANGE_THRESHOLD_SEMITONES
        time_contiguous = (t - prev_time) <= gap_threshold
        if pitch_close and time_contiguous:
            seg_midi_values.append(midi)
        else:
            segments.append((seg_start_time, prev_time, seg_midi_values))
            seg_start_time = t
            seg_midi_values = [midi]
        prev_time = t
    segments.append((seg_start_time, prev_time, seg_midi_values))

    cleaned = []
    for start, end, midi_values in segments:
        duration = end - start
        avg_midi = round(sum(midi_values) / len(midi_values))
        note_name = librosa.midi_to_note(avg_midi)
        if cleaned:
            prev_start, prev_end, prev_note, prev_midi = cleaned[-1]
            gap = start - prev_end
            is_short_glitch = duration < MIN_SEGMENT_DURATION
            is_same_note_reunion = (note_name == prev_note) and (gap <= gap_threshold)
            if is_short_glitch or is_same_note_reunion:
                cleaned[-1] = (prev_start, end, prev_note, prev_midi)
                continue
        cleaned.append((start, end, note_name, avg_midi))

    return [{"start": round(s, 3), "end": round(e, 3), "note": n, "midi": m} for s, e, n, m in cleaned]

print("Note extraction function ready.")

In [ ]:
import shutil

output_dir = Path("output") / song_id
output_dir.mkdir(parents=True, exist_ok=True)

stem_files = {}
for wav_file in stems_source_dir.glob("*.wav"):
    stem_name = wav_file.stem
    dest = output_dir / f"{stem_name}.wav"
    shutil.copy(wav_file, dest)
    stem_files[stem_name] = dest
    print(f"Copied {stem_name}")

notes_available = []
for stem_name in FREQ_RANGES:
    if stem_name not in stem_files:
        continue
    print(f"Extracting notes for {stem_name}...")
    timeline = extract_note_timeline(str(stem_files[stem_name]), stem_name)
    (output_dir / f"notes_{stem_name}.json").write_text(json.dumps(timeline))
    notes_available.append(stem_name)
    print(f"  {len(timeline)} note segments found")

manifest = {
    "song_id": song_id,
    "stems": list(stem_files.keys()),
    "notes_available": notes_available,
}
(output_dir / "manifest.json").write_text(json.dumps(manifest))
print("\nManifest written:", manifest)

## Download

Extract the downloaded zip so that `<song_id>/manifest.json` ends up
directly inside your local project's `backend/data/`, e.g.
`backend/data/my_song/manifest.json` -- not nested inside an extra
`output/` folder. The app scans `backend/data/*/manifest.json` on startup.

In [ ]:
import shutil
from google.colab import files as colab_files

zip_path = shutil.make_archive(song_id, "zip", root_dir="output", base_dir=song_id)
print(f"Zipped: {zip_path}")
colab_files.download(zip_path)